# Chapter 4 — Text Classification
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)

---

Text classification is one of the most common NLP tasks. This chapter compares **five approaches** across two model families:

| Approach | Model type | Labels needed? | Expected accuracy |
|---|---|---|---|
| Task-specific model | Representation | No (pre-trained) | 80% |
| Embeddings + LogisticRegression | Representation | Yes | 85% |
| Averaged class embeddings | Representation | Yes (for averaging) | 84% |
| Zero-shot label embeddings | Representation | No | 78% |
| Flan-T5 generation | Generative | No | 84% |
| ChatGPT API | Generative | No | 91% |

---

## Table of Contents

- [Part 1: Data and Evaluation Metrics](#part-1-data-and-evaluation-metrics)
  - [Exercise 1.1 — Load and Explore the Dataset](#exercise-11--load-and-explore-the-dataset)
  - [Exercise 1.2 — Confusion Matrix From Scratch](#exercise-12--confusion-matrix-from-scratch)
  - [Exercise 1.3 — Precision, Recall, F1 From Scratch](#exercise-13--precision-recall-f1-from-scratch)
  - [Exercise 1.4 — Build the evaluate_performance Helper](#exercise-14--build-the-evaluate_performance-helper)
- [Part 2: Task-Specific Model](#part-2-task-specific-model)
  - [Exercise 2.1 — Load and Run Inference](#exercise-21--load-and-run-inference)
  - [Exercise 2.2 — Evaluate and Observe Domain Mismatch](#exercise-22--evaluate-and-observe-domain-mismatch)
- [Part 3: Classification with Embeddings](#part-3-classification-with-embeddings)
  - [3A: Supervised — Embeddings + LogisticRegression](#3a-supervised--embeddings--logisticregression)
  - [3B: Averaged Class Embeddings](#3b-averaged-class-embeddings)
  - [3C: Zero-Shot with Label Descriptions](#3c-zero-shot-with-label-descriptions)
- [Part 4: Generative Models](#part-4-generative-models)
  - [4A: Flan-T5](#4a-flan-t5)
  - [4B: ChatGPT API](#4b-chatgpt-api)
- [Part 5: Final Comparison](#part-5-final-comparison)

In [ ]:
# %%capture
# !pip install "transformers==4.41.2" "sentence-transformers==3.0.1" openai
# !pip install -U datasets scikit-learn

---
# Part 1: Data and Evaluation Metrics

Before building any classifier, we need two things: data and a way to measure performance. This part loads the **Rotten Tomatoes** movie review dataset and implements the evaluation metrics from scratch so you understand exactly what sklearn computes for you.

## Exercise 1.1 — Load and Explore the Dataset

The **Rotten Tomatoes** dataset contains 5,331 positive and 5,331 negative short movie reviews. Labels: `0 = negative`, `1 = positive`.

**Task:**
1. Load `"rotten_tomatoes"` from HuggingFace datasets
2. Print the dataset structure (splits, sizes, features)
3. Print the **first** and **last** example from the training set
4. Print the label distribution in the test set (how many 0s vs 1s)

In [ ]:
from datasets import load_dataset

# YOUR CODE HERE
# data = load_dataset("rotten_tomatoes")
# print(data)

# Print first and last training example
# print("First training example:", data["train"][0])
# print("Last training example:", data["train"][-1])

# Label distribution in test set
# labels = data["test"]["label"]
# print(f"\nTest set: {labels.count(0)} negative, {labels.count(1)} positive")

## Exercise 1.2 — Confusion Matrix From Scratch

A **confusion matrix** breaks down predictions into four categories:

```
                Predicted Positive   Predicted Negative
Actual Positive   True Positive (TP)   False Negative (FN)
Actual Negative   False Positive (FP)  True Negative (TN)
```

**Task:** Implement `confusion_matrix_values(y_true, y_pred)` that returns `(TP, FP, TN, FN)`. Test it on this toy example:

```python
y_true = [1, 1, 0, 1, 0, 0, 1, 0]
y_pred = [1, 0, 0, 1, 1, 0, 1, 0]
# Expected: TP=3, FP=1, TN=3, FN=1
```

In [ ]:
def confusion_matrix_values(y_true: list, y_pred: list) -> tuple:
    """Return (TP, FP, TN, FN) for binary classification."""
    # YOUR CODE HERE
    # TP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    # FP = ...
    # TN = ...
    # FN = ...
    pass


y_true = [1, 1, 0, 1, 0, 0, 1, 0]
y_pred = [1, 0, 0, 1, 1, 0, 1, 0]

# tp, fp, tn, fn = confusion_matrix_values(y_true, y_pred)
# print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")
# Expected: TP=3, FP=1, TN=3, FN=1

## Exercise 1.3 — Precision, Recall, F1 From Scratch

The four classification metrics:

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of all predicted positive, how many were correct?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of all actual positive, how many did we find?)}$$

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(harmonic mean of precision and recall)}$$

$$\text{Accuracy} = \frac{TP + TN}{TP + FP + TN + FN} \quad \text{(overall fraction correct)}$$

**Task:** Implement all four metrics using the output of `confusion_matrix_values`. Verify they match `sklearn.metrics.classification_report`.

**Expected for the toy example:** Precision=0.75, Recall=0.75, F1=0.75, Accuracy=0.75

In [ ]:
def precision(tp, fp): 
    # YOUR CODE HERE
    pass

def recall(tp, fn): 
    # YOUR CODE HERE
    pass

def f1_score(prec, rec): 
    # YOUR CODE HERE
    pass

def accuracy(tp, fp, tn, fn): 
    # YOUR CODE HERE
    pass


# tp, fp, tn, fn = confusion_matrix_values(y_true, y_pred)
# prec = precision(tp, fp)
# rec  = recall(tp, fn)
# f1   = f1_score(prec, rec)
# acc  = accuracy(tp, fp, tn, fn)

# print(f"Precision: {prec:.4f}")
# print(f"Recall:    {rec:.4f}")
# print(f"F1:        {f1:.4f}")
# print(f"Accuracy:  {acc:.4f}")

# Verify with sklearn
# from sklearn.metrics import classification_report
# print(classification_report(y_true, y_pred))

## Exercise 1.4 — Build the `evaluate_performance` Helper

This function is reused by every part of the notebook.

**Task:** Implement `evaluate_performance(y_true, y_pred)` that prints a sklearn `classification_report` with target names `["Negative Review", "Positive Review"]`.

In [ ]:
from sklearn.metrics import classification_report


def evaluate_performance(y_true, y_pred):
    """Print classification report with Negative/Positive labels."""
    # YOUR CODE HERE
    # performance = classification_report(
    #     y_true, y_pred,
    #     target_names=["Negative Review", "Positive Review"]
    # )
    # print(performance)
    pass


# Quick sanity check on toy data
# evaluate_performance(y_true, y_pred)

---
# Part 2: Task-Specific Model

The most direct approach: use a model that was already fine-tuned on a sentiment classification task. We use **Twitter-RoBERTa**, a RoBERTa model fine-tuned on ~124 million tweets for sentiment analysis.

This model was not trained on movie reviews — it was trained on tweets. This is an example of **domain mismatch**: the training domain (social media) differs from the target domain (film criticism). We will observe its effect on performance.

💡 **GPU required** for this part.

## Exercise 2.1 — Load and Run Inference

**Task:**
1. Load `"cardiffnlp/twitter-roberta-base-sentiment-latest"` into a HuggingFace pipeline with `return_all_scores=True` on `device="cuda:0"`
2. Run inference on the full test set using `KeyDataset(data["test"], "text")`
3. For each output, extract `output[0]["score"]` (negative) and `output[2]["score"]` (positive)
4. Use `np.argmax` to pick the predicted class, append to `y_pred`

*Note: `return_all_scores=True` gives scores for all 3 classes: [negative, neutral, positive]*

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# YOUR CODE HERE
# pipe = pipeline(
#     model=model_path,
#     tokenizer=model_path,
#     return_all_scores=True,
#     device="cuda:0"
# )

# y_pred = []
# for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
#     negative_score = output[0]["score"]
#     positive_score = output[2]["score"]
#     assignment = np.argmax([negative_score, positive_score])
#     y_pred.append(assignment)

## Exercise 2.2 — Evaluate and Observe Domain Mismatch

**Task:** Evaluate performance using `evaluate_performance`. Then answer in the observation cell:
- Which class has lower recall — positive or negative?
- Why might a tweet-trained model struggle on movie reviews?

**Expected accuracy: ~80%**

In [ ]:
# YOUR CODE HERE
# evaluate_performance(data["test"]["label"], y_pred)

**Observations:**

- Which class has lower recall? *(write here)*
- Why does domain mismatch (tweets vs movie reviews) hurt performance? *(write here)*
- What would you do to improve this without retraining? *(write here)*

---
# Part 3: Classification with Embeddings

Instead of a task-specific classifier head, we use a general-purpose **embedding model** to convert each review into a 768-dimensional vector. Then we classify in embedding space.

Three strategies, in order of decreasing label requirements:
1. **Supervised** — train a logistic regression on labelled embeddings
2. **Averaged class embeddings** — average each class's embeddings, classify by cosine similarity
3. **Zero-shot** — embed the label descriptions and classify by cosine similarity (no labelled examples)

### Encode All Reviews

All three strategies start by encoding the train and test sets. We do this once and reuse the embeddings.

**Task:** Load `sentence-transformers/all-mpnet-base-v2` and encode both splits. Print the embedding shapes.

In [ ]:
from sentence_transformers import SentenceTransformer

# YOUR CODE HERE
# model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
# test_embeddings  = model.encode(data["test"]["text"],  show_progress_bar=True)

# print(f"Train embeddings shape: {train_embeddings.shape}")
# print(f"Test embeddings shape:  {test_embeddings.shape}")
# Expected: (8530, 768) and (1066, 768)

## 3A: Supervised — Embeddings + LogisticRegression

The embeddings are features. A logistic regression learns a decision boundary in this 768-dimensional space. This is a two-step pipeline:

```
Text  →  [Embedding model (frozen)]  →  768-dim vector  →  [LogisticRegression]  →  0 or 1
```

**Task:**
1. Train a `LogisticRegression(random_state=42)` on `train_embeddings` and `data["train"]["label"]`
2. Predict on `test_embeddings`
3. Evaluate performance

**Expected accuracy: ~85%** — better than the task-specific model despite using a generic embedding model.

In [ ]:
from sklearn.linear_model import LogisticRegression

# YOUR CODE HERE
# clf = LogisticRegression(random_state=42)
# clf.fit(train_embeddings, data["train"]["label"])

# y_pred = clf.predict(test_embeddings)
# evaluate_performance(data["test"]["label"], y_pred)

## 3B: Averaged Class Embeddings

What if we skip the classifier entirely? We average all training embeddings per class to get one "prototype" vector per class. Then we classify test examples by finding the closest prototype.

```
Class 0 prototype = mean of all negative review embeddings
Class 1 prototype = mean of all positive review embeddings

Test example → cosine_similarity(test_emb, [prototype_0, prototype_1]) → argmax → predicted class
```

**Task:**
1. Create a DataFrame by stacking `train_embeddings` with the label column
2. Group by label and compute the mean embedding per class → `averaged_target_embeddings` shape: `(2, 768)`
3. Compute cosine similarity between all test embeddings and the two prototypes
4. Predict using `np.argmax(sim_matrix, axis=1)`
5. Evaluate

**Expected accuracy: ~84%** — nearly identical to logistic regression with no training at all.

**Hint:**
```python
df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
averaged_target_embeddings = df.groupby(768).mean().values  # column 768 is the label
```

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# YOUR CODE HERE
# Step 1 & 2: Average embeddings per class
# df = pd.DataFrame(np.hstack([train_embeddings, np.array(data["train"]["label"]).reshape(-1, 1)]))
# averaged_target_embeddings = df.groupby(768).mean().values
# print(f"Prototype shapes: {averaged_target_embeddings.shape}")

# Step 3 & 4: Cosine similarity and prediction
# sim_matrix = cosine_similarity(test_embeddings, averaged_target_embeddings)
# y_pred = np.argmax(sim_matrix, axis=1)

# Step 5: Evaluate
# evaluate_performance(data["test"]["label"], y_pred)

## 3C: Zero-Shot with Label Descriptions

What if we have **no labelled data at all**? We describe what each class means in plain English, embed those descriptions, and classify test examples by which description they are closest to.

```
"A negative review"  →  [embedding model]  →  label vector 0
"A positive review"  →  [embedding model]  →  label vector 1
test review          →  [embedding model]  →  test vector
  → cosine_similarity(test_vector, [label_0, label_1]) → argmax → predicted class
```

**No training data required.** The model generalises purely from the semantic meaning of the label descriptions.

### Exercise 3C.1 — Basic Zero-Shot

**Task:**
1. Encode the label descriptions: `["A negative review", "A positive review"]`
2. Compute cosine similarity between `test_embeddings` and `label_embeddings`
3. Predict with `np.argmax` and evaluate

**Expected accuracy: ~78%**

In [ ]:
# YOUR CODE HERE
# label_embeddings = model.encode(["A negative review", "A positive review"])
# sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
# y_pred = np.argmax(sim_matrix, axis=1)
# evaluate_performance(data["test"]["label"], y_pred)

### Exercise 3C.2 — Label Engineering Experiment

The wording of label descriptions significantly affects zero-shot accuracy. More specific descriptions give the embedding model more signal to work with.

**Task:** Run the same zero-shot pipeline with each of these three label description pairs and compare the accuracies:

```python
label_sets = [
    ["A negative review", "A positive review"],                        # baseline
    ["A very negative movie review", "A very positive movie review"],   # movie-specific
    ["This film was terrible", "This film was excellent"],             # example-based
]
```

**Observe:** Which label wording gives the highest accuracy? Why might more specific descriptions help?

In [ ]:
from sklearn.metrics import accuracy_score

label_sets = [
    ["A negative review", "A positive review"],
    ["A very negative movie review", "A very positive movie review"],
    ["This film was terrible", "This film was excellent"],
]

# YOUR CODE HERE
# For each label_set:
#   encode label descriptions
#   compute cosine similarity
#   predict and compute accuracy
#   print label_set and accuracy

# for labels in label_sets:
#     label_embs = model.encode(labels)
#     sim = cosine_similarity(test_embeddings, label_embs)
#     preds = np.argmax(sim, axis=1)
#     acc = accuracy_score(data["test"]["label"], preds)
#     print(f"{labels}  →  accuracy: {acc:.4f}")

---
# Part 4: Generative Models

Generative models approach classification differently: instead of outputting a class index (0 or 1), they generate text. We extract the class from that text.

```
Representation model:  "Best movie ever" → model → 1
Generative model:      "Is this positive or negative? Best movie ever" → model → "positive" → 1
```

## 4A: Flan-T5

**Flan-T5** is an encoder-decoder model fine-tuned on hundreds of NLP tasks using natural language instructions. It takes text as input and generates text as output — even for classification tasks.

The classification prompt is simply prepended to the review:
```
"Is the following sentence positive or negative? " + review_text
```

The model outputs `"positive"` or `"negative"`, which we map to 1 or 0.

### Exercise 4A.1 — Inspect the T5 Architecture

Unlike the decoder-only Phi-3 from Chapter 3, Flan-T5 is an **encoder-decoder** model.

**Task:**
1. Load `google/flan-t5-small` into a `text2text-generation` pipeline
2. Print `pipe.model` to inspect the architecture
3. Answer: how many encoder blocks and how many decoder blocks does flan-t5-small have?

In [ ]:
# YOUR CODE HERE
# pipe = pipeline(
#     "text2text-generation",
#     model="google/flan-t5-small",
#     device="cuda:0"
# )
# print(pipe.model)

**Observation:** How many encoder blocks and decoder blocks does flan-t5-small have? *(write here)*

### Exercise 4A.2 — Run Classification

**Task:**
1. Prepend the prompt to every example: `data = data.map(lambda example: {"t5": prompt + example['text']})`
2. Run inference over `data["test"]` using `KeyDataset`
3. Map `"negative"` → 0 and anything else → 1
4. Evaluate

**Expected accuracy: ~84%**

In [ ]:
# YOUR CODE HERE
# prompt = "Is the following sentence positive or negative? "
# data = data.map(lambda example: {"t5": prompt + example['text']})

# y_pred = []
# for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
#     text = output[0]["generated_text"]
#     y_pred.append(0 if text == "negative" else 1)

# evaluate_performance(data["test"]["label"], y_pred)

## 4B: ChatGPT API

Closed-source models like GPT-3.5 are accessed via API. They require no local GPU — inference runs on OpenAI's servers. The trade-off: cost per token and latency.

**⚠️ Note:** This exercise requires an OpenAI API key. If you don't have one, read through the code and note the expected result (91% accuracy). You can run it with a few examples to verify the function works.

**Prompt design principle:** Be explicit about the output format. Tell the model exactly what to return (`0` or `1`), otherwise it may generate explanations instead of a clean label.

In [ ]:
# import openai
# client = openai.OpenAI(api_key="YOUR_KEY_HERE")

### Exercise 4B.1 — Implement the Generation Function

**Task:** Implement `chatgpt_generation(prompt, document, model)` that:
1. Creates a messages list with a system message and a user message
2. Replaces `[DOCUMENT]` in the prompt with the actual document text
3. Calls `client.chat.completions.create` with `temperature=0`
4. Returns the generated text string

In [ ]:
def chatgpt_generation(prompt: str, document: str, model: str = "gpt-3.5-turbo-0125") -> str:
    """Generate a classification label using ChatGPT."""
    # YOUR CODE HERE
    # messages = [
    #     {"role": "system", "content": "You are a helpful assistant."},
    #     {"role": "user",   "content": prompt.replace("[DOCUMENT]", document)}
    # ]
    # chat_completion = client.chat.completions.create(
    #     messages=messages,
    #     model=model,
    #     temperature=0
    # )
    # return chat_completion.choices[0].message.content
    pass


# Test with one example
prompt_template = """Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

# document = "unpretentious , charming , quirky , original"
# result = chatgpt_generation(prompt_template, document)
# print(f"Result: {result}")  # Expected: '1'

### Exercise 4B.2 — Run on Full Test Set

**⚠️ Only run this if you have API credits.** It makes 1,066 API calls.

**Task:** Call `chatgpt_generation` for every review in `data["test"]["text"]`, collect predictions, convert to int, and evaluate.

**Expected accuracy: ~91%** — highest of all approaches.

In [ ]:
# YOUR CODE HERE — skip if you want to save API credits
# predictions = [chatgpt_generation(prompt_template, doc) for doc in tqdm(data["test"]["text"])]
# y_pred = [int(pred) for pred in predictions]
# evaluate_performance(data["test"]["label"], y_pred)

---
# Part 5: Final Comparison

Fill in the results table below after running each part.

## Results Summary

| # | Approach | Model | Labels needed | GPU | Accuracy |
|---|---|---|---|---|---|
| 1 | Task-specific model | Twitter-RoBERTa | No | Yes | *(fill in)* |
| 2 | Embeddings + LogReg | all-mpnet-base-v2 | Yes | Yes (encode) | *(fill in)* |
| 3 | Averaged class embeddings | all-mpnet-base-v2 | Yes (for avg) | Yes (encode) | *(fill in)* |
| 4 | Zero-shot label embeddings | all-mpnet-base-v2 | No | Yes (encode) | *(fill in)* |
| 5 | Flan-T5 generation | flan-t5-small | No | Yes | *(fill in)* |
| 6 | ChatGPT API | gpt-3.5-turbo | No | No (API) | *(fill in)* |

## Reflection Questions

Answer these after completing all parts:

1. **No labels available:** Which approach would you use first? *(zero-shot, Flan-T5, or ChatGPT?)*

2. **Labels available, need best accuracy:** Which approach wins?

3. **Production deployment (fast, cheap, no API cost):** Which approach fits best?

4. **The embedding + LogReg approach beats the task-specific model (85% vs 80%).** Why might a general embedding model outperform a sentiment-specific model in this case?

5. **Zero-shot got 78% with no labels.** What single change most improved zero-shot accuracy in Exercise 3C.2?

---
## Chapter 4 Summary

| Concept | What you implemented | Key insight |
|---|---|---|
| Evaluation metrics | TP/FP/TN/FN, Precision, Recall, F1, Accuracy from scratch | Classification report is just arithmetic over the confusion matrix |
| Task-specific model | Twitter-RoBERTa → 80% | Domain mismatch (tweets ≠ film reviews) hurts performance |
| Supervised embeddings | all-mpnet + LogReg → 85% | General embeddings + simple classifier beats task-specific model |
| Averaged embeddings | Prototype vectors → 84% | Near-identical to LogReg with zero training step |
| Zero-shot | Label descriptions → 78% | Label wording matters — more specific = higher accuracy |
| Flan-T5 | Text-to-text → 84% | Encoder-decoder; prompt becomes the task instruction |
| ChatGPT API | GPT-3.5 → 91% | Best accuracy, but costs money and requires internet access |